# GPU arm for Thabat

Runs the CPU versus GPU comparison on a free Kaggle or Colab GPU. It runs two arms on the same machine, `ref_k` (CPU, 1 thread, float32) and `gpu` (CUDA, float32, TF32 off), for seeds 0 to 59 of the three RL tasks. The analysis compares `gpu` with `ref_k`, so the machine is held fixed.

Before you start: switch on a GPU in the session settings, and switch on internet access (needed for git and pip). The run can be stopped and started again, finished seeds are skipped. A full run is long, so it is fine to stop early: the analysis uses the seeds that are finished and reports how many there are.

When it is done, download the archive made in the last cell, copy the `*.kaggle.jsonl` and `*.kaggle.envs.json` files into `results/raw/` of the repository, and commit them. Until then the GPU predictions stay NOT RUN.

In [ ]:
!git clone https://github.com/hudasol/thabat
%cd thabat
!pip install -q gymnasium mujoco
!pip install -q -e . --no-deps

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU is attached to this session"
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
# Order is chosen so that a stopped run still has complete tasks.
for task in ["pendulum", "cartpole", "invpend"]:
    !python scripts/run_task.py --task {task} --arms ref_k,gpu --seeds 0:60 --shard kaggle

In [ ]:
import glob, shutil
files = glob.glob("results/raw/*.kaggle.*")
print(len(files), "files")
shutil.make_archive("/kaggle/working/thabat_gpu_arm" if __import__("os").path.isdir("/kaggle/working") else "thabat_gpu_arm", "zip", "results/raw")